# Score-Based Generative Modeling on 2D GMM

This notebook generates the two key visuals for the poster:

- **Visual 1 — Sampling Dynamics:** Scatter plots of generated samples from the three reverse samplers (Euler-Maruyama, Probability Flow ODE, Predictor-Corrector), each with the ground-truth $p_0$ density in the background.
- **Visual 2 — Convergence Analysis:** Sliced Wasserstein distance vs. Number of Function Evaluations (NFE) for three method combinations.

Toggle `USE_EXACT_SCORE` to switch between the analytic score and the trained MLP.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import yaml
import torch

from data.gaussian_mixture import gaussian_mix
from models.solver import euler_maruyama, probability_flow_ode, predictor_corrector
from models.score_model import ScoreNet
from utils.wasserstein import sliced_wasserstein

## Configuration

Load all hyperparameters from `config/vp_config.yaml` and build the GMM.

In [2]:
with open('../config/vp_config.yaml') as f:
    cfg = yaml.safe_load(f)

np.random.seed(cfg['seed'])

# noise schedule
beta_min = cfg['noise_scheduler']['beta_min']
beta_max = cfg['noise_scheduler']['beta_max']
T        = cfg['noise_scheduler']['T']

# inference
t_eps      = cfg['inference']['t_eps']
n_steps    = cfg['inference']['n_steps']
n_samples  = cfg['inference']['n_samples']
n_corrector = cfg['inference']['n_corrector']
snr        = cfg['inference']['snr']

# plot
vis_n   = cfg['plot']['vis_n']
xy_lim  = cfg['plot']['xy_lim']
n_levels = cfg['plot']['n_levels']

# build GMM
K      = cfg['data']['gmm']['K']
R      = cfg['data']['gmm']['R']
sigma0 = cfg['data']['gmm']['sigma']
angles = np.linspace(0, 2 * np.pi, K, endpoint=False)
mus    = np.stack([R * np.cos(angles), R * np.sin(angles)], axis=1)
sigmas = np.full((K, 2), sigma0)
gmm    = gaussian_mix(mus, sigmas)

# time grid for reverse sampling (T -> t_eps)
ts = np.linspace(T, t_eps, n_steps + 1)

print(f'GMM: {K} components, R={R}, sigma={sigma0}')
print(f'Reverse steps: {n_steps},  t in [{t_eps}, {T}]')

GMM: 8 components, R=5, sigma=0.25
Reverse steps: 500,  t in [0.001, 1.0]


## Score Function

Set `USE_EXACT_SCORE = True` to use the analytic score $\nabla_x \log p_t(x)$ derived from the closed-form VP-SDE perturbation kernel.

Set `USE_EXACT_SCORE = False` to load the trained ScoreNet MLP from `models/score_net.pt`.

Both are wrapped into the same interface `score_fn(x: ndarray, t: float) -> ndarray`.

In [3]:
USE_EXACT_SCORE = True  # <-- toggle here

if USE_EXACT_SCORE:
    score_fn = lambda x, t: gmm.exact_score(x, t, beta_min, beta_max)
    score_label = 'Exact Score'
else:
    train_cfg = cfg['training']
    model = ScoreNet(
        data_dim     = 2,
        hidden_dim   = train_cfg['hidden_dim'],
        n_layers     = train_cfg['n_layers'],
        time_emb_dim = train_cfg['time_emb_dim'],
        min_freq     = train_cfg['min_freq'],
        max_freq     = train_cfg['max_freq'],
    )
    model.load_state_dict(torch.load('../models/score_net.pt', map_location='cpu'))
    model.eval()
    score_fn = model.score_fn
    score_label = 'Fitted Score (MLP)'

print(f'Using: {score_label}')

Using: Exact Score


## Visual 1 — Sampling Dynamics

Run all three reverse samplers from $x_T \sim \mathcal{N}(0, I)$ and plot the final samples against the ground-truth $p_0$ density contours.

| Sampler | Equation | Stochastic? |
|---|---|---|
| **P** (Euler-Maruyama) | Reverse VP-SDE, EM discretization | Yes |
| **ODE** (Probability Flow) | Deterministic ODE with same marginals | No |
| **PC** (Predictor-Corrector) | EM predictor + Langevin corrector | Yes |

In [4]:
x_T = np.random.randn(n_samples, 2)

print('Running Euler-Maruyama...')
traj_em  = euler_maruyama(score_fn, x_T, ts, beta_min, beta_max)
samples_em = traj_em[-1]

print('Running Probability Flow ODE...')
traj_ode = probability_flow_ode(score_fn, x_T, ts, beta_min, beta_max)
samples_ode = traj_ode[-1]

print('Running Predictor-Corrector...')
traj_pc  = predictor_corrector(score_fn, x_T, ts, beta_min, beta_max,
                               n_corrector=n_corrector, snr=snr)
samples_pc = traj_pc[-1]

print('Done.')

Running Euler-Maruyama...
Running Probability Flow ODE...
Running Predictor-Corrector...
Done.


In [ ]:
# precompute density grid for background
x_grid = np.linspace(-xy_lim, xy_lim, vis_n)
y_grid = np.linspace(-xy_lim, xy_lim, vis_n)
X, Y   = np.meshgrid(x_grid, y_grid)
pts    = np.stack([X.ravel(), Y.ravel()], axis=1)
Z0     = gmm.density(pts).reshape(vis_n, vis_n)

def plot_density_bg(ax, Z, color='steelblue', n_lev=n_levels):
    """Draw smooth filled density contours from transparent to opaque."""
    z_ceil  = Z.max() * 1.001
    levels  = np.linspace(Z.max() * 0.05, Z.max(), n_lev)
    alphas  = np.linspace(0.04, 1.0, n_lev)
    for i in range(len(levels) - 1):
        ax.contourf(X, Y, Z, levels=[levels[i], levels[i+1]],
                    colors=[color], alpha=alphas[i])
    ax.contourf(X, Y, Z, levels=[levels[-1], z_ceil],
                colors=[color], alpha=alphas[-1])

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
configs = [
    (samples_em,  'P Sampler\n(Euler-Maruyama)'),
    (samples_pc,  'PC Sampler\n(Predictor-Corrector)'),
    (samples_ode, 'ODE Sampler\n(Probability Flow)'),
]

for ax, (samples, title) in zip(axes, configs):
    plot_density_bg(ax, Z0, color='steelblue')
    ax.scatter(samples[:, 0], samples[:, 1],
               s=4, alpha=0.4, color='white', linewidths=0)
    ax.set_xlim(-xy_lim, xy_lim)
    ax.set_ylim(-xy_lim, xy_lim)
    ax.set_title(f'{title}\n({score_label})', fontsize=10)
    ax.set_aspect('equal')
    ax.set_xlabel(r'$x_1$')
    ax.set_ylabel(r'$x_2$')

plt.tight_layout()
out_path = '../reference/visual1_dynamics.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved to {out_path}')
plt.show()

## Visual 2 — Convergence Analysis

Sweep over NFE (Number of Function Evaluations) and measure the **Sliced Wasserstein Distance** between generated samples and true GMM samples.

Three curves are plotted:
1. **Exact Score + P Sampler** — baseline stochastic reverse SDE
2. **Exact Score + PC Sampler** — corrector reduces discretization error
3. **Fitted Score + PC Sampler** — approximation error floor from MLP

For the PC sampler with $n_c$ corrector steps per predictor step, NFE $= n_{\text{steps}} \times (1 + n_c)$,
so $n_{\text{steps}} = \lfloor \text{NFE} / (1 + n_c) \rfloor$.

In [ ]:
# reference samples from the true GMM
REF_N   = 5000
x_ref   = gmm.sample(REF_N)

NFE_LIST = [10, 25, 50, 100, 250, 500, 1000]
N_SLICES = 200   # projections for sliced Wasserstein
SW_SEED  = 42

# exact score closure (always available for curve 1 & 2)
exact_score_fn = lambda x, t: gmm.exact_score(x, t, beta_min, beta_max)

# fitted score (load regardless of USE_EXACT_SCORE for curve 3)
train_cfg = cfg['training']
_fitted_model = ScoreNet(
    data_dim     = 2,
    hidden_dim   = train_cfg['hidden_dim'],
    n_layers     = train_cfg['n_layers'],
    time_emb_dim = train_cfg['time_emb_dim'],
    min_freq     = train_cfg['min_freq'],
    max_freq     = train_cfg['max_freq'],
)
_fitted_model.load_state_dict(torch.load('../models/score_net.pt', map_location='cpu'))
_fitted_model.eval()
fitted_score_fn = _fitted_model.score_fn

print('Reference and score functions ready.')

In [ ]:
def run_sweep(score_fn, sampler, nfe_list, nc=1):
    """Run sampler at each NFE budget and return sliced Wasserstein distances."""
    swd_list = []
    for nfe in nfe_list:
        if sampler == 'p':
            steps = nfe
        else:  # pc: each step costs 1 (predictor) + nc (corrector)
            steps = max(1, nfe // (1 + nc))

        ts_sweep = np.linspace(T, t_eps, steps + 1)
        x_init   = np.random.randn(n_samples, 2)

        if sampler == 'p':
            traj = euler_maruyama(score_fn, x_init, ts_sweep, beta_min, beta_max)
        else:
            traj = predictor_corrector(score_fn, x_init, ts_sweep, beta_min, beta_max,
                                       n_corrector=nc, snr=snr)
        swd = sliced_wasserstein(traj[-1], x_ref, n_slices=N_SLICES, seed=SW_SEED)
        swd_list.append(swd)
        print(f'  NFE={nfe:5d}  steps={steps:5d}  SWD={swd:.4f}')
    return swd_list

print('=== Exact Score + P Sampler ===')
swd_p_exact = run_sweep(exact_score_fn, 'p', NFE_LIST)

print('\n=== Exact Score + PC Sampler ===')
swd_pc_exact = run_sweep(exact_score_fn, 'pc', NFE_LIST, nc=n_corrector)

print('\n=== Fitted Score + PC Sampler ===')
swd_pc_fitted = run_sweep(fitted_score_fn, 'pc', NFE_LIST, nc=n_corrector)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(NFE_LIST, swd_p_exact,   'o-',  label='Exact Score + P Sampler',        color='steelblue')
ax.plot(NFE_LIST, swd_pc_exact,  's-',  label='Exact Score + PC Sampler',        color='darkorange')
ax.plot(NFE_LIST, swd_pc_fitted, '^--', label='Fitted Score (MLP) + PC Sampler', color='firebrick')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Function Evaluations (NFE)', fontsize=11)
ax.set_ylabel('Sliced Wasserstein Distance', fontsize=11)
ax.set_title('Convergence Analysis: SWD vs NFE', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
out_path = '../reference/visual2_convergence.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved to {out_path}')
plt.show()